# DATA-445: In-Class Assignment 14

Consider the `framingham.csv` data file. The dataset is available on the Kaggle website, and it is from an ongoing cardiovascular study on residents of the town of Framingham, Massachusetts. The classification goal is to predict whether the patient has 10-year risk of future coronary heart disease (CHD). The dataset provides the patients' information. It includes over 4,000 records and 15 attributes. Each attribute is a potential risk factor. There are both demographic, behavioral and medical risk factors.

**Demographic:**
- Sex: male or female (Nominal)
- Age: Age of the patient (Continuous - Although the recorded ages have been truncated to whole numbers, the concept of age is continuous)

**Behavioral:**
- Current Smoker: whether or not the patient is a current smoker (Nominal)
- Cigs Per Day: the number of cigarettes that the person smoked on average in one day (can be considered continuous as one can have any number of cigarettes, even half a cigarette.)

**Medical (history):**
- BP Meds: whether or not the patient was on blood pressure medication (Nominal)
- Prevalent Stroke: whether or not the patient had previously had a stroke (Nominal)
- Prevalent Hyp: whether or not the patient was hypertensive (Nominal)
- Diabetes: whether or not the patient had diabetes (Nominal)

**Medical (current):**
- Tot Chol: total cholesterol level (Continuous)
- Sys BP: systolic blood pressure (Continuous)
- Dia BP: diastolic blood pressure (Continuous)
- BMI: Body Mass Index (Continuous)
- Heart Rate: heart rate (Continuous - In medical research, variables such as heart rate though in fact discrete, yet are considered continuous because of large number of possible values.)
- Glucose: glucose level (Continuous)

**Predict variable (desired target):**
- 10 year risk of coronary heart disease CHD (binary: "1" means "Yes", "0" means "No")

In **Python**, answer the following:

## 1. (4 points) 
Using the `pandas` library, read the csv data file and create a data-frame called `heart`. Remove the missing values.

In [1]:
import pandas as pd 

heart = pd.read_csv("framingham.csv")
heart = heart.dropna()
heart.head()

,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,TenYearCHD
0,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0


## 2. (4 points) 
Using `age`, `currentSmoker`, `totChol`, `BMI`, and `heartRate` as the predictor variables, and `TenYearCHD` is the target variable. Define the 5-fold cross-validation strategy.

In [2]:

from sklearn.model_selection import StratifiedKFold

X = heart[["age", "currentSmoker", "totChol", "BMI", "heartRate"]]
y = heart["TenYearCHD"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 3. (8 points) 
Train and evaluate an `ExtraTreesClassifier` model (with `n_estimators = 500`, and `max_depth = 3`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [3]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import recall_score

et_md = ExtraTreesClassifier(n_estimators=500,
                             max_depth=3,
                             random_state=42)

et_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    et_md.fit(X_train, y_train)
    y_pred = et_md.predict_proba(X_test)[:, 1]
    score = recall_score(y_test, (y_pred > 0.1).astype(int))
    et_recalls.append(score)
    
    print(f"Fold {i+1} recall: {score}")
    
print(f"Mean recall: {sum(et_recalls)/len(et_recalls)}")

Fold 1 recall: 0.9821428571428571
Fold 2 recall: 0.9819819819819819
Fold 3 recall: 1.0
Fold 4 recall: 1.0
Fold 5 recall: 0.9732142857142857
Mean recall: 0.9874678249678249


## 4. (8 points) 
Train and evaluate a `GradientBoostingClassifier` model (with `n_estimators = 500` and `max_depth = 3`, and `learning_rate = 0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [4]:
from sklearn.ensemble import GradientBoostingClassifier

gb_md = GradientBoostingClassifier(n_estimators=500,
                                   max_depth=3,
                                   learning_rate=0.01,
                                   random_state=42)

gb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    gb_md.fit(X_train, y_train)
    y_pred = gb_md.predict_proba(X_test)[:, 1]
    score = recall_score(y_test, (y_pred > 0.1).astype(int))
    gb_recalls.append(score)
    
    print(f"Fold {i+1} recall: {score}")

print(f"Mean recall: {sum(gb_recalls)/len(gb_recalls)}")

Fold 1 recall: 0.7053571428571429
Fold 2 recall: 0.8468468468468469
Fold 3 recall: 0.918918918918919
Fold 4 recall: 0.8558558558558559
Fold 5 recall: 0.8303571428571429
Mean recall: 0.8314671814671815


## 5. (8 points) 
Train and evaluate a `HistGradientBoostingClassifier` model (with `max_iter = 500` and `max_depth = 3`, and `learning_rate = 0.01`) over the five-fold cross-validation strategy defined in part 2. Using 10% as a cutoff value, report the average recall over the cross-validation strategy.

In [5]:
from sklearn.ensemble import HistGradientBoostingClassifier

hgb_md = HistGradientBoostingClassifier(max_iter=500,
                                        max_depth=3,
                                        learning_rate=0.01,
                                        random_state=42)

hgb_recalls = []
for i, (train_index, test_index) in enumerate(skf.split(X, y)):
    
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    hgb_md.fit(X_train, y_train)
    y_pred = hgb_md.predict_proba(X_test)[:, 1]
    score = recall_score(y_test, (y_pred > 0.1).astype(int))
    hgb_recalls.append(score)
    
    print(f"Fold {i+1} recall: {score}")

print(f"Mean recall: {sum(hgb_recalls)/len(hgb_recalls)}")

Fold 1 recall: 0.75
Fold 2 recall: 0.8468468468468469
Fold 3 recall: 0.9279279279279279
Fold 4 recall: 0.8918918918918919
Fold 5 recall: 0.8303571428571429
Mean recall: 0.8494047619047619


## 6. (3 points) 
Using the results from parts (3), (4), and (5), what model would you use to predict `TenYearCHD` based on the recall?

based on the findings of the models I would use the ExtraTreesClassifier to predict the TenYearCHD because the average recall is higher